In [ ]:
print('==== THE ANALYSIS TABLE ======')

In [ ]:
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path
from binance.client import Client
from matplotlib import pyplot as plt
from datetime import datetime

# --------------------------------
import plotly.graph_objects as go
import plotly.express as px
# --------------  configure df printing layout -------- to avoid chopping --
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)
# pd.set_option('display.width', 1000)
#--------------------------------------------------------------------------

# import aggtrades
cwd = Path.cwd()
# cwd = Path(__file__).resolve()
file_path = cwd/'__input_ag'/'SOLUSDT-aggTrades-2026-07-08.parquet'
# symbol name
SYMBOL = file_path.name.split("-")[0]
# file_output_path = cwd.parent/'__output_ag'/f"{SYMBOL}_ouput_sample.csv"
file_output_path = cwd.parent/'__output_ag'/f"{SYMBOL}_ouput_sample.parquet"

# chart naming
fname = datetime.now().strftime("%Y%m%d_%H%M%S")
chart1_path = cwd.parent.parent/'__charts'/f'regime_{fname}.png'

# print(file_path.exists())

In [ ]:
# coin stats like average volume from BINANCE

client = Client()
klines = client.futures_klines(symbol=SYMBOL, interval='1d', limit=21)
closed_klines = klines[:-1]  # Exclude today's incomplete candle
# Index 4: Close Price, Index 5: Volume
volumes = [float(k[5]) for k in closed_klines]
prices = [float(k[4]) for k in closed_klines]
# ---------------------------------
vol_20_ma = sum(volumes[-20:]) / 20
price_20_ma = sum(prices[-20:]) / 20
notional_ma = vol_20_ma * price_20_ma
# print(vol_20_ma)  

In [ ]:
# to stop the program incase notional_ma is not available or 0
print(f"\n volume ma for {SYMBOL} : $ {notional_ma:,.2f} \n")


# load csv to df
# df = pd.read_csv(file_path)
df = pd.read_parquet(file_path)

# rename columns

df = df.rename(columns={'agg_trade_id': 'agg_tde_id', 'quantity': 'base_qty',
                        'first_trade_id': 'first_tde_id', 'last_trade_id': 'last_tde_id'
                        })

# format some columns -- keep time as datetime , not string for GROUPING
# .dt.strftime("%Y-%m-%d %H:%M:%S")
df['transact_time'] = pd.to_datetime(df['transact_time'], unit='ms')
# helper columns
df['quote_qty_usdt'] = df['price'] * df['base_qty']
df['buy_vol_usdt'] = np.where(
    df['is_buyer_maker'] == False, df['quote_qty_usdt'], 0.0)
df['sell_vol_usdt'] = np.where(
    df['is_buyer_maker'] == True, df['quote_qty_usdt'], 0.0)

# BUCKETING TIME WINDOWS --- groupby --- aggregating

bdf = df.groupby(pd.Grouper(key='transact_time', freq='5Min')).agg(tde1_price=('price', 'first'), last_price=(
    'price', 'last'), buyVol_usdt=('buy_vol_usdt', 'sum'), sellVol_usdt=('sell_vol_usdt', 'sum')).reset_index()
# rename columns
bdf = bdf.rename(columns={'transact_time': 'time'})
# derived columns  ------------------------------------- PERCENTAGES
bdf['delta'] = (bdf['buyVol_usdt'] - bdf['sellVol_usdt']).round(3)
bdf['prc_change_window'] = ((
    bdf['last_price']-bdf['tde1_price'])/bdf['tde1_price'])*100
day_open_price = bdf['tde1_price'].iloc[0]
bdf['dayOpen_prcchange'] = ((
    bdf['last_price'] - day_open_price)/day_open_price)*100

# core derived columns
bdf['s_delta'] = (bdf['delta']/notional_ma)*100
bdf['cumulative_sd'] = bdf['s_delta'].cumsum()

bdf['mkt_eff'] = (np.where(bdf['s_delta'] != 0,
                    (bdf['dayOpen_prcchange']/bdf['s_delta']), 0.00))
bdf['mkt_eff'] = (bdf['mkt_eff']*100).abs()

bdf['mkt_regime'] = (np.where(bdf['cumulative_sd'] != 0,
                        (bdf['dayOpen_prcchange']/bdf['cumulative_sd']), 0.00))
bdf['mkt_regime'] = (bdf['mkt_regime']*100).abs()

# ----------COLUMN FORMATING ------------------------
bdf['delta'] = bdf['delta'] #.round(2)
bdf['buyVol_usdt'] = bdf['buyVol_usdt']#.round(2)
bdf['sellVol_usdt'] = bdf['sellVol_usdt']#.round(2)

# bdf.to_parquet(file_output_path)

print(bdf.tail())




In [ ]:
# print(" all is well ")
# print(bdf.tail())

# ------------ GRAPHS ANALYSIS  -------------------------------

# fig1 = px.line(bdf, x='time', y='buyVol_usdt')
# fig1.show()

# fig2 = px.line(bdf, x='time', y='delta')
# fig2.show()

# print(bdf.columns.tolist())
# print(bdf.head())

low_eff = bdf.sort_values(by='s_delta', ascending=False)
# print(low_eff.head())
print(bdf['s_delta'].head())


In [ ]:
#- -----------  TREND GRAPH --------------

# fig1=px.line(bdf, x='time', y='dayOpen_prcchange', title='MARKET TREND')
fig1= go.Figure()
fig1.add_trace(go.Scatter(x=bdf.time, y=bdf.dayOpen_prcchange, mode='lines', line=dict(color='purple'), name='change 4rm day_opn' ))
# fig.update_layout(title= 'TREND GRAPH')
fig1.show()

# -------------- PRESSURE FLOW GRAPH -----------------
# fig3= px.line(bdf, x='time', y='cumulative_sd', title='MARKET PRESSURE FLOW')
# fig3.add_trace(go.Scatter(x=bdf.time, y=bdf.cumulative_sd, mode='lines', line=dict(color='green'), name='cum % delta'))

# fig3.add_hline(y=0)

In [ ]:
#- -----------  MARKET PRESSURE GRAPH --------------

# fig2 =px.line(bdf, x='time', y='s_delta', title='MARKET PRESSURE')

fig2= go.Figure()
fig2.add_trace(go.Scatter(x=bdf.time, y=bdf.s_delta, mode='lines', line=dict(color='blue'), name='% delta' ))
fig2.add_trace(go.Scatter(x=bdf.time, y=bdf.prc_change_window, mode='lines', line=dict(color="#dd4508"), name='5m prc change' ))
# fig2.update_layout(title= 'MARKET PRESSURE GRAPH' )

In [ ]:
#- -----------  MARKET PRESSURE FLOW GRAPH --------------
# fig3= px.line(bdf, x='time', y='cumulative_sd', title='MARKET PRESSURE FLOW')
fig3= go.Figure()
fig3.add_trace(go.Scatter(x=bdf.time, y=bdf.cumulative_sd, mode='lines+markers', line=dict(color='green'), name='cumulative pctg delta'))
# fig3.update_layout(title= 'MARKET PRESSURE FLOW GRAPH' )

In [ ]:
# --------------------- OUTLIER GRAPHS
# -------------- MARKET EFFICIENCY GRAPH -----------------
# fig4= px.line(bdf, x='time', y='mkt_eff', title='MARKET efficiency GRAPH')
fig4= go.Figure()
fig4.add_trace(go.Scatter(x=bdf.time, y=bdf.mkt_eff, mode='lines', line=dict(color='black'), name='mkt eff'))

# -------------- MARKET price  GRAPH -----------------
# fig6= px.line(bdf, x='time', y='tde1_price', title='price')
fig6= go.Figure()
fig6.add_trace(go.Scatter(x=bdf.time, y=bdf.tde1_price, mode='lines', line=dict(), name='first price'))
# fig6.show()

# -------------- MARKET REGIME STRUCTURE GRAPH -----------------
# fig5= px.line(bdf, x='time', y='mkt_regime', title='MARKET structure')
fig5= go.Figure()
fig5.add_trace(go.Scatter(x=bdf.time, y=bdf.mkt_regime, mode='lines', line=dict(color='black'), name='mkt regime'))